In [5]:
import os
import re
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer

# Path that contains GitHub repositories
REFERENCE_PATH = "data/reference_corpus/"

# List to hold cleaned code
clean_texts = []


Removes comments and extra whitespace from Python code.
This helps reduce noise for embeddings.

In [6]:
def clean_code(code):
    """
    Cleans Python code by:
    - Removing single-line comments (# ...)
    - Removing multi-line comments ('''  ''' or """  """)
    - Removing extra spaces and newlines
    """
    # Remove multi-line comments ("""...""" or '''...''')
    code = re.sub(r'("""[\s\S]*?"""|\'\'\'[\s\S]*?\'\'\')', '', code)

    # Remove single-line comments
    code = re.sub(r'#.*', '', code)

    # Remove extra spaces/newlines
    code = re.sub(r'\s+', ' ', code).strip()

    return code


Collects and cleans Python files from all folders inside reference_corpus


In [7]:
for folder in sorted(os.listdir(REFERENCE_PATH)):
    folder_path = os.path.join(REFERENCE_PATH, folder)
    
    if os.path.isdir(folder_path):  # Ensure it is a folder
        print(f"Processing folder: {folder_path}")

        for root, _, files in os.walk(folder_path):
            for filename in files:
                if filename.endswith(".py"):
                    try:
                        filepath = os.path.join(root, filename)
                        with open(filepath, "r", encoding="utf-8") as f:
                            raw_code = f.read()
                            cleaned = clean_code(raw_code)

                            if len(cleaned) > 10:  # Skip empty/small files
                                clean_texts.append(cleaned)
                    except Exception as e:
                        print(f"Error reading {filepath}: {e}")

print(f"Successfully loaded {len(clean_texts)} Python files.")


Processing folder: data/reference_corpus/1
Processing folder: data/reference_corpus/2
Processing folder: data/reference_corpus/3
Processing folder: data/reference_corpus/4
Processing folder: data/reference_corpus/5
Successfully loaded 148 Python files.


Make sure indexes folder exists if not create, also Save cleaned code for reference and Save embeddings file

In [8]:
os.makedirs("indexes", exist_ok=True)

with open("indexes/clean_texts.pkl", "wb") as f:
    pickle.dump(clean_texts, f)

print("Saved cleaned code to indexes/clean_texts.pkl")

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(clean_texts, convert_to_numpy=True)

np.save("indexes/embeddings.npy", embeddings)

print("Saved embeddings.npy to indexes/ folder")
print(f"Embedding shape: {embeddings.shape}")

# Preview some results
print("Number of cleaned snippets:", len(clean_texts))
print("\nFirst 200 characters of first cleaned file:\n")
print(clean_texts[0][:200])


Saved cleaned code to indexes/clean_texts.pkl
Saved embeddings.npy to indexes/ folder
Embedding shape: (148, 384)
Number of cleaned snippets: 148

First 200 characters of first cleaned file:

from __future__ import annotations from itertools import combinations def combination_lists(n: int, k: int) -> list[list[int]]: return [list(x) for x in combinations(range(1, n + 1), k)] def generate_
